# FIFA World Cup 2026 Tournament Simulator - Real Data Pipeline

This notebook trains on historical match data, tunes multiple models, evaluates on a holdout test set, and simulates the official 2026 World Cup groups with Monte Carlo runs. No synthetic training data is used.

## What this notebook does

1. Mounts Google Drive and clones the repository
2. Downloads the real Kaggle football datasets
3. Uses `results.csv` to train the model and derive rankings with Elo
4. Tunes multiple model families with cross-validation
5. Evaluates the winning model on a holdout test set
6. Simulates the official 2026 World Cup groups and displays the results

In [52]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import shutil

PROJECT_DIR = Path('/content/world-cup-2026-simulator')
DRIVE_DIR = Path('/content/drive/MyDrive/world-cup-2026-simulator')
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
MODELS_DIR = PROJECT_DIR / 'models'
DRIVE_RAW_DIR = DRIVE_DIR / 'data' / 'raw'
DRIVE_MODELS_DIR = DRIVE_DIR / 'models'
RAW_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['WORLD_CUP_REQUIRE_REAL_DATA'] = '1'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Clone or update the repository in Colab.
if not PROJECT_DIR.exists():
    !git clone https://github.com/musa-qureshi/world-cup-2026-simulator.git /content/world-cup-2026-simulator
else:
    %cd /content/world-cup-2026-simulator
    !git pull

%cd /content/world-cup-2026-simulator

In [ ]:
# Install dependencies.
!pip -q install -r requirements.txt
!pip -q install kaggle

## Download real data

This notebook uses the historical football dataset from Kaggle. The key file is `results.csv`, and the notebook also keeps `goalscorers.csv`, `shootouts.csv`, and `former_names.csv` available for analysis and future feature work.

In [ ]:
# Authenticate Kaggle using kaggle.json from either /content or Google Drive.
kaggle_token = Path('/content/kaggle.json')
if not kaggle_token.exists():
    drive_token = Path('/content/drive/MyDrive/kaggle.json')
    if drive_token.exists():
        shutil.copy2(drive_token, kaggle_token)

if not kaggle_token.exists():
    raise FileNotFoundError('kaggle.json not found. Upload it to Colab or place it in Google Drive at MyDrive/kaggle.json.')

!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d martj42/international-football-results-from-1872-to-2017 -p /content/world-cup-2026-simulator/data/raw --unzip

In [ ]:
# Validate the downloaded real files before training.
required_files = ['results.csv', 'goalscorers.csv', 'shootouts.csv', 'former_names.csv']
missing_files = [name for name in required_files if not (RAW_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing required Kaggle files: {missing_files}')

print('Downloaded files:')
for name in sorted(os.listdir(RAW_DIR)):
    print('-', name)

In [ ]:
# Prepare the files expected by the training pipeline and derive rankings from the real match history.
import pandas as pd
from src.elo_rating import EloRatingSystem
from src.preprocessing import load_match_data

matches = load_match_data(RAW_DIR / 'results.csv')
matches.to_csv(RAW_DIR / 'matches.csv', index=False)
matches.to_csv(RAW_DIR / 'international_results.csv', index=False)

elo = EloRatingSystem().bulk_fit(matches)
rankings = elo.to_frame().sort_values('rating', ascending=False).reset_index(drop=True)
rankings.insert(1, 'rank', range(1, len(rankings) + 1))
rankings['confederation'] = 'Unknown'
rankings = rankings[['team', 'rank', 'rating', 'confederation']]
rankings.to_csv(RAW_DIR / 'fifa_rankings.csv', index=False)
rankings.to_csv(RAW_DIR / 'rankings.csv', index=False)

actual_groups = {
    'Group A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'Group B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'Group C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'Group D': ['United States', 'Paraguay', 'Australia', 'Turkey'],
    'Group E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'],
    'Group F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'Group G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'Group H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'Group I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'Group J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'Group K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'Group L': ['England', 'Croatia', 'Ghana', 'Panama'],
}
team_rows = [{'group': group, 'team': team} for group, teams in actual_groups.items() for team in teams]
pd.DataFrame(team_rows).to_csv(RAW_DIR / 'world_cup_2026_teams.csv', index=False)
pd.DataFrame([{'group': group, 'team': team} for group, teams in actual_groups.items() for team in teams]).to_csv(RAW_DIR / 'world_cup_2026_groups.csv', index=False)

print('Prepared real training files and derived rankings from actual results.')
print('Matches rows:', len(matches))
print('Rankings rows:', len(rankings))

In [ ]:
# Confirm the real inputs are complete and visible.
for filename in ['matches.csv', 'fifa_rankings.csv', 'goalscorers.csv', 'shootouts.csv', 'former_names.csv']:
    path = RAW_DIR / filename
    print(filename, 'exists:', path.exists())

matches_check = pd.read_csv(RAW_DIR / 'matches.csv')
required_match_columns = ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
missing_match_columns = [column for column in required_match_columns if column not in matches_check.columns]
if missing_match_columns:
    raise ValueError(f'Matches file is missing required columns: {missing_match_columns}')

print('Matches shape:', matches_check.shape)
print('Matches columns OK.')

In [61]:
# Refresh the repository before training so Colab uses the latest code from disk.
%cd /content/world-cup-2026-simulator
!git pull --ff-only
!git rev-parse --short HEAD

/content/world-cup-2026-simulator
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 10.13 KiB | 3.38 MiB/s, done.
From https://github.com/musa-qureshi/world-cup-2026-simulator
   3b14381..7850e56  main       -> origin/main
Updating 3b14381..7850e56
Fast-forward
 colab_world_cup_2026_real_pipeline.ipynb | 1431 +++++++++++++++++++-----------
 1 file changed, 917 insertions(+), 514 deletions(-)
7850e56


In [53]:
# Patch the training module in Colab so the notebook works even if the repo clone is stale.
import src.train_model as train_model
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None


def _parameter_grids(random_state: int = 42) -> dict[str, dict[str, list]]:
    grids: dict[str, dict[str, list]] = {
        'logistic_regression': {
            'model__C': [0.25, 0.5, 1.0, 2.0],
            'model__solver': ['lbfgs'],
        },
        'random_forest': {
            'model__n_estimators': [300, 500],
            'model__max_depth': [None, 10, 20],
            'model__min_samples_split': [2, 4],
            'model__min_samples_leaf': [1, 2],
        },
    }
    if XGBClassifier is not None:
        grids['xgboost'] = {
            'model__n_estimators': [150, 300],
            'model__max_depth': [3, 4, 5],
            'model__learning_rate': [0.03, 0.05, 0.1],
            'model__subsample': [0.8, 0.9],
            'model__colsample_bytree': [0.8, 0.9],
        }
    if LGBMClassifier is not None:
        grids['lightgbm'] = {
            'model__n_estimators': [200, 400],
            'model__num_leaves': [31, 63],
            'model__learning_rate': [0.03, 0.05],
            'model__subsample': [0.8, 0.9],
            'model__colsample_bytree': [0.8, 0.9],
        }
    return grids


def _tune_model(name, pipeline, X_train, y_train, random_state: int = 42):
    grids = _parameter_grids(random_state)
    param_grid = grids.get(name, {})
    if not param_grid:
        pipeline.fit(X_train, y_train)
        return pipeline, {'cv_best_score': None, 'best_params': {}, 'n_candidates': 1}

    class_counts = y_train.value_counts()
    min_class = int(class_counts.min()) if not class_counts.empty else 2
    n_splits = max(3, min(5, min_class))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring='neg_log_loss',
        cv=cv,
        n_jobs=-1,
        refit=True,
        error_score='raise',
    )
    search.fit(X_train, y_train)
    summary = {
        'cv_best_score': float(search.best_score_),
        'best_params': search.best_params_,
        'n_candidates': int(len(search.cv_results_['params'])),
    }
    return search.best_estimator_, summary

train_model._tune_model = _tune_model
train_model._parameter_grids = _parameter_grids
print('Patched src.train_model with tuning helpers inside Colab.')

Patched src.train_model with tuning helpers inside Colab.


In [62]:
# Patch simulator knockout handling in Colab to support stale repo clones.
from src.simulator import MatchOutcome, WorldCupSimulator


def _patched_simulate_knockout_round(self, pairings):
    if not pairings:
        return []

    first = pairings[0]
    if isinstance(first, MatchOutcome):
        winners = [match.winner for match in pairings if match.winner is not None]
        pairings = [(winners[i], winners[i + 1]) for i in range(0, len(winners), 2)]

    return [self.simulate_match(home_team, away_team, knockout=True) for home_team, away_team in pairings]


WorldCupSimulator._simulate_knockout_round = _patched_simulate_knockout_round
print('Patched WorldCupSimulator._simulate_knockout_round for Colab runtime compatibility.')

Patched WorldCupSimulator._simulate_knockout_round for Colab runtime compatibility.


In [55]:
# Train with cross-validated hyperparameter tuning and evaluate on a holdout test set.
from src.train_model import train_and_save_model

training_result = train_and_save_model()
print('Best model:', training_result.best_model_name)
print('Model saved to:', training_result.model_path)
print('Metadata saved to:', training_result.metadata_path)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014002 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4625
[LightGBM] [Info] Number of data points in the train set: 39429, number of used features: 27
[LightGBM] [Info] Start training from score -0.715174
[LightGBM] [Info] Start training from score -1.475612
[LightGBM] [Info] Start training from score -1.264947
Best model: xgboost
Model saved to: /content/world-cup-2026-simulator/models/match_prediction_model.joblib
Metadata saved to: /content/world-cup-2026-simulator/models/match_prediction_metadata.json


In [56]:
# Summarize the tuned model comparison and holdout performance.
import pandas as pd
from src.utils import read_json

metadata = read_json(training_result.metadata_path)
metrics = metadata.get('metrics', {})
summary_rows = []
for model_name, result in metrics.items():
    summary_rows.append({
        'model': model_name,
        'cv_best_score': result.get('cv_best_score'),
        'best_params': result.get('best_params'),
        'accuracy': result.get('accuracy'),
        'log_loss': result.get('log_loss'),
        'f1_score': result.get('f1_score'),
        'n_candidates': result.get('n_candidates'),
    })
summary_frame = pd.DataFrame(summary_rows).sort_values('log_loss', ascending=True)
display(summary_frame)

best_model_metrics = metrics[metadata['best_model_name']]
print('Best model:', metadata['best_model_name'])
print('Holdout accuracy:', best_model_metrics['accuracy'])
print('Holdout log loss:', best_model_metrics['log_loss'])
print('Holdout macro F1:', best_model_metrics['f1_score'])

,model,cv_best_score,best_params,accuracy,log_loss,f1_score,n_candidates
2,xgboost,-0.885203,"{'model__colsample_bytree': 0.8, 'model__learn...",0.586630,0.883651,0.430358,72
3,lightgbm,-0.886643,"{'model__colsample_bytree': 0.8, 'model__learn...",0.585920,0.885034,0.434641,32
1,random_forest,-0.897476,"{'model__max_depth': None, 'model__min_samples...",0.583688,0.894631,0.456517,24
0,logistic_regression,-0.934710,"{'model__C': 0.25, 'model__solver': 'lbfgs'}",0.547880,0.931739,0.513448,4


Best model: xgboost
Holdout accuracy: 0.5866301481030635
Holdout log loss: 0.8836507397264103
Holdout macro F1: 0.4303581456838444


In [57]:
# Load the tuned predictor and show a sample real-team prediction.
from src.predict import MatchPredictor

predictor = MatchPredictor()
sample_prediction = predictor.predict_match('France', 'Brazil')
print(sample_prediction.probabilities.as_dict())
print('Expected goals:', sample_prediction.expected_home_goals, sample_prediction.expected_away_goals)
print('Model:', sample_prediction.model_name)

{'team_a': 0.4309743046760559, 'draw': 0.3035244941711426, 'team_b': 0.2655012011528015}
Expected goals: 1.5137329274434395 1.07026486398445
Model: xgboost


In [ ]:
# Refresh the repository before training so Colab uses the latest code from disk.
%cd /content/world-cup-2026-simulator
!git pull --ff-only
!git rev-parse --short HEAD

## Monte Carlo simulation for the official 2026 World Cup draw

The notebook uses the actual 2026 groups and simulates the tournament many times to produce championship, semifinal, finalist, and elimination probabilities.

In [60]:
%%bash
cd /content/world-cup-2026-simulator
ls -lah models
ls -lah /content/drive/MyDrive/world-cup-2026-simulator/models

total 2.7M
drwxr-xr-x 2 root root 4.0K May 29 17:35 .
drwxr-xr-x 8 root root 4.0K May 29 17:52 ..
-rw-r--r-- 1 root root 420K May 29 17:35 match_prediction_metadata.json
-rw-r--r-- 1 root root 2.3M May 29 17:35 match_prediction_model.joblib
total 2.7M
-rw------- 1 root root 420K May 29 17:52 match_prediction_metadata.json
-rw------- 1 root root 2.3M May 29 17:52 match_prediction_model.joblib


In [63]:
from src.simulator import WorldCupSimulator

simulator = WorldCupSimulator(predictor=predictor, groups=actual_groups)
simulation_summary = simulator.simulate_many_tournaments(n_simulations=5000)
print('Monte Carlo runs completed: 5000')

Monte Carlo runs completed: 5000


In [64]:
# Display the final simulation results in tables and charts.
import pandas as pd
import plotly.express as px

champion_frame = pd.DataFrame(sorted(simulation_summary.champion_probabilities.items(), key=lambda item: item[1], reverse=True), columns=['team', 'champion_probability'])
finalist_frame = pd.DataFrame(sorted(simulation_summary.final_probabilities.items(), key=lambda item: item[1], reverse=True), columns=['team', 'final_probability'])
semifinal_frame = pd.DataFrame(sorted(simulation_summary.semifinal_probabilities.items(), key=lambda item: item[1], reverse=True), columns=['team', 'semifinal_probability'])
elimination_frame = pd.DataFrame(sorted(simulation_summary.group_elimination_probabilities.items(), key=lambda item: item[1], reverse=True), columns=['team', 'group_elimination_probability'])
finishing_frame = pd.DataFrame(sorted(simulation_summary.average_finishing_position.items(), key=lambda item: item[1]))
finishing_frame.columns = ['team', 'average_finishing_position']

display(champion_frame.head(20))
display(finalist_frame.head(20))
display(semifinal_frame.head(20))
display(elimination_frame.head(20))
display(finishing_frame.head(20))

fig1 = px.bar(champion_frame.head(16), x='team', y='champion_probability', title='2026 World Cup Title Odds')
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()

stage_frame = pd.DataFrame({
    'team': champion_frame['team'],
    'champion': champion_frame['champion_probability'],
    'final': [simulation_summary.final_probabilities.get(team, 0.0) for team in champion_frame['team']],
    'semifinal': [simulation_summary.semifinal_probabilities.get(team, 0.0) for team in champion_frame['team']],
    'group_elimination': [simulation_summary.group_elimination_probabilities.get(team, 0.0) for team in champion_frame['team']],
})
heatmap = stage_frame.head(24).set_index('team').T
fig2 = px.imshow(heatmap, aspect='auto', color_continuous_scale='Blues', title='Tournament Stage Probabilities')
fig2.show()

,team,champion_probability
0,Spain,0.1368
1,Argentina,0.0976
2,France,0.0966
3,Brazil,0.0572
4,England,0.0522
5,Portugal,0.0440
6,Netherlands,0.0430
7,Colombia,0.0402
8,Germany,0.0402
9,Croatia,0.0342


,team,final_probability
0,Spain,0.3990
1,Argentina,0.3012
2,France,0.2958
3,Brazil,0.2108
4,England,0.1964
5,Netherlands,0.1776
6,Portugal,0.1708
7,Germany,0.1702
8,Colombia,0.1600
9,Croatia,0.1430


,team,semifinal_probability
0,Spain,0.5682
1,Argentina,0.4600
2,France,0.4524
3,Brazil,0.3556
4,England,0.3444
5,Netherlands,0.3166
6,Germany,0.3140
7,Portugal,0.3050
8,Colombia,0.2954
9,Morocco,0.2730


,team,group_elimination_probability
0,Curaçao,0.7338
1,Qatar,0.7126
2,Ghana,0.6864
3,Haiti,0.6842
4,Iraq,0.6662
5,Saudi Arabia,0.6340
6,Jordan,0.6034
7,Cape Verde,0.5782
8,Tunisia,0.5766
9,New Zealand,0.5734


,team,average_finishing_position
0,None,3.0000
1,Spain,9.9824
2,Argentina,11.7508
3,France,11.7772
4,England,12.4562
5,Brazil,12.5092
6,Germany,12.6016
7,Switzerland,13.1726
8,Netherlands,13.3448
9,Belgium,13.6494


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [66]:
# Show one example simulated tournament bracket summary.
tournament = simulator.simulate_tournament()
print('Champion:', tournament['champion'])
print('Runner-up:', tournament['runner_up'])
print('Third place:', tournament['third_place'])
print('Finalists:', tournament['finalists'])
print('Semifinalists:', tournament['semifinalists'])

Champion: Belgium
Runner-up: Australia
Third place: Argentina
Finalists: ['Belgium', 'Australia', 'Ecuador', 'Portugal']
Semifinalists: ['Argentina', 'Australia', 'Ecuador', 'Portugal', 'Belgium', 'Colombia', 'Panama', 'Sweden']


## Notes

- This notebook uses only real historical football data for training.
- The model is tuned with cross-validation and selected on holdout performance.
- The 2026 simulation uses the official groups drawn for the tournament.
- If you want, I can also wire this notebook into the Streamlit dashboard so the same tuned model and 2026 outputs appear in the app.